# Chekc baseline results
Why is the mean baseline getting a high pearson per cell, and what does BABEL add?

We use existing predictions and scores made by the OpenProblems benchmark on the validation set.

## 1. Bestanden

In [ ]:
from pathlib import Path
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from IPython.display import display

ROOT = Path('/home/michiel/thesis/benchmark/task_predict_modality_upstream_validation')
BASE = ROOT / 'outputs/tchic/runs/h3k4me3_genelevel_dev/mean_per_gene/work'
paths = {
    'target': ROOT / 'outputs/tchic/prepared/h3k4me3_genelevel_dev/test_mod2.h5ad',
    'Mean': BASE / 'ba/6c6a3bd813ded280bdc61f4b8fdccf/run.mean_per_gene.mean_per_gene.output.h5ad',
    'BABEL': ROOT / 'outputs/tchic/runs/babel_20epochs/babel_prediction.h5ad',
}
score_paths = {
    'Mean': BASE / '44/a7225954451bdd5b6a757320f5d845/run.mean_per_gene.correlation.correlation.output.h5ad',
    'BABEL': ROOT / 'outputs/tchic/evaluations/h3k4me3_genelevel_dev/official_001/babel_20epochs/correlation/metric/publish/score.h5ad',
}
OUT = ROOT / 'outputs/tchic/diagnostics/h3k4me3_genelevel_dev'
OUT.mkdir(parents=True, exist_ok=True)

def show(name):
    plt.tight_layout()
    plt.savefig(OUT / f'{name}.png', dpi=160, bbox_inches='tight')
    plt.show()

def dense(x):
    return x.toarray().astype(float) if sparse.issparse(x) else np.asarray(x, dtype=float)


## 2. Loading predictions and official scores
 
We controleren alleen of de bestanden bij dezelfde cellen en genen horen. Niet-eindige voorspellingen worden, net als in de officiële metric, als nul behandeld.

In [ ]:
data = {name: ad.read_h5ad(path) for name, path in paths.items()}
target = data['target']
scores = {name: ad.read_h5ad(path) for name, path in score_paths.items()}
for name, obj in {**data, **{f'score_{k}': v for k, v in scores.items()}}.items():
    assert obj.obs_names.equals(target.obs_names), f'Cellen/volgorde verschillen: {name}'
    assert obj.var_names.equals(target.var_names), f'Genen/volgorde verschillen: {name}'
    assert obj.uns['dataset_id'] == target.uns['dataset_id'], name
assert target.obs_names.is_unique and target.var_names.is_unique

matrices = {k: sparse.csr_matrix(v.layers['normalized'], dtype=float) for k, v in data.items()}
assert np.isfinite(matrices['target'].data).all()
for name in scores:
    values = matrices[name].data
    bad = ~np.isfinite(values)
    if bad.any():
        print(f'{name}: {bad.sum()} niet-eindige predictions als nul behandeld.')
        values[bad] = 0
Y = matrices['target']
P = {name: matrices[name] for name in scores}
r = pd.DataFrame({name: obj.obs['pearson'].to_numpy() for name, obj in scores.items()},
                 index=target.obs_names)
assert np.isfinite(r.to_numpy()).all()
for name, obj in scores.items():
    official = dict(zip(obj.uns['metric_ids'], obj.uns['metric_values']))
    assert np.isclose(r[name].mean(), official['mean_pearson_per_cell'])
display(r.describe().loc[['mean', 'std', 'min', '50%', 'max']])
print(f'{target.n_obs:,} cellen × {target.n_vars:,} genen')


## 3. Is BABEL better for the same cells?

Left is a distribution of the Pearson scores.

Right is one point per cell, above the diagonal, BABEL is better

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for name, color in [('Mean', '#0072B2'), ('BABEL', '#E69F00')]:
    ax[0].hist(r[name], bins=np.linspace(0, 1, 51), histtype='step', lw=2, label=name, color=color)
ax[0].set(xlabel='Officiële Pearson per cel', ylabel='Aantal cellen')
ax[0].legend()
ax[1].scatter(r['Mean'], r['BABEL'], s=8, alpha=.3, color='#0072B2')
# ax[1].plot([-1, 1], [-1, 1], '--', color='black', lw=1)
ax[1].plot([0.225, 0.85], [0.225, 0.85], '--', color='black', lw=1)
ax[1].set(xlabel='Mean: Pearson per cel', ylabel='BABEL: Pearson per cel')
show('01_per_cell_pearson')
print(f"BABEL scoort hoger in {(r.BABEL > r.Mean).mean():.1%} van de cellen.")

The distributions of the pearson scores mostly overlap, but the BABEL one seems to be a little more to the right.

The points are close to the diagonal. Cells that get a higher score for the baseline often get a high score for BABEL aswell. 

## 4. What is behind the correlation?
We kiezen voorbeelden van lage, middelste en hoge scores.

Elk punt is één gen. Dezelfde assen gelden voor beide modellen. De stippellijn is een perfecte voorspelling. Een hoge Pearson vereist niet dat punten op die lijn liggen.

In [ ]:
order = np.argsort(r['Mean'].to_numpy())
selected = order[np.rint(np.array([0, .1, .5, .9, 1]) * (len(order) - 1)).astype(int)]
fig, axes = plt.subplots(5, 2, figsize=(15, 18))
for row, i in enumerate(selected):
    total_cell_count = float(target.layers["counts"][i].sum())
    y = dense(Y[i]).ravel()
    limit = max(y.max(), *(dense(p[i]).max() for p in P.values())) * 1.03
    for col, (name, p) in enumerate(P.items()):
        ax = axes[row, col]
        ax.scatter(y, dense(p[i]).ravel(), s=3, alpha=.2, color='#0072B2', rasterized=True)
        ax.plot([0, limit], [0, limit], '--', color='black', lw=1)
        ax.set(xlim=(-0.1, limit), ylim=(0, limit), xlabel='Gemeten RNA (log_cp10k)',
               ylabel='Voorspeld RNA (log_cp10k)',
               title=(
                   f'{name} | cel {i} | officiële r = {r[name].iloc[i]:.3f}\n'
                   f'Totaal RNA-counts: {total_cell_count:,.0f}'
               )
              )
        ax.set_aspect('equal')
show('02_expression_scatter')
display(pd.DataFrame({'cel': target.obs_names[selected], 'positie': selected}))


A higher total measured RNA-count seems to give a higher pearson correlation score. 0.228 at 799 counts and 0.841 at 25177 counts.

The vertical lines are there because raw counts are whole numbers. In a cell this those numbers gets a certain log-CP10k score which will always be the same for value. (cell 389: ln(1 + 10000/799) = 2.6)

A lot of genes have a measurement of 0 RNA, for which a positive expression is predicted. On the other side, for a lot of high measured genes the predicion is lower.

BABEL doesnt score higher for every cell.

## 5. Which cells add most to the pearson of the mean per gene baseline?

In [ ]:
scores = r["Mean"].to_numpy()
order = np.argsort(scores)[::-1]  # Hoogste Pearson eerst

display(pd.DataFrame({
    "cel": target.obs_names,
    "Pearson": scores,
}).nlargest(10, "Pearson"))

results = []
for n in [0, 10, 100, 500, 1000]:
    if n < len(scores):
        results.append({
            "Weggelaten cellen": n,
            "Overgebleven cellen": len(scores) - n,
            "Gemiddelde Pearson": scores[order[n:]].mean(),
        })

display(pd.DataFrame(results))

## 6. Which genes add most to the pearson of the mean per gene baseline?

In [ ]:
def unit_rows(a):
    centered = a - a.mean(axis=1, keepdims=True)
    norm = np.linalg.norm(centered, axis=1, keepdims=True)
    return np.divide(centered, norm, out=np.zeros_like(centered), where=norm > 0)

contribution = np.zeros(target.n_vars)
for start in range(0, target.n_obs, 128):
    sl = slice(start, start + 128)
    contribution += (unit_rows(dense(Y[sl])) * unit_rows(dense(P['Mean'][sl]))).sum(axis=0)
contribution /= target.n_obs
assert np.isclose(contribution.sum(), r['Mean'].mean(), atol=1e-5)
genes = pd.DataFrame({'gene': target.var_names, 'Pearson_bijdrage': contribution})
display(genes.nlargest(10, 'Pearson_bijdrage'))
keep = np.ones(target.n_vars, dtype=bool)
keep[np.argsort(contribution)[-10:]] = False
reduced = []
for start in range(0, target.n_obs, 128):
    sl = slice(start, start + 128)
    reduced.extend((unit_rows(dense(Y[sl][:, keep])) *
                    unit_rows(dense(P['Mean'][sl][:, keep]))).sum(axis=1))
display(pd.Series({'Officiële baseline-Pearson': r['Mean'].mean(),
                   'Diagnostiek: zonder top 10 genen': np.mean(reduced)}))
genes.to_csv(OUT / 'baseline_gene_contributions.csv', index=False)


Without the 10 genes with the biggest positive contribution, the score drops with 0.0075, so the high score does not just come from the 10 highest genes.

## 7. Hangen de scores samen met RNA-detectie of depth?

Elk punt is een cel. Het aantal gedetecteerde genen wordt uit de gemeten normalized-laag gehaald. Als counts beschikbaar is, plotten we ook de som van die counts per cel. Dit is de depth van de beschikbare genenmatrix, niet noodzakelijk alle oorspronkelijke reads.

In [ ]:
qc = pd.DataFrame({'Gedetecteerde RNA-genen': np.asarray((Y > 0).sum(axis=1)).ravel()},
                  index=target.obs_names)
if 'counts' in target.layers:
    qc['RNA-counts in genenmatrix'] = np.asarray(target.layers['counts'].sum(axis=1)).ravel()
else:
    print("Geen counts-laag: de depth-plot wordt overgeslagen.")
fig, axes = plt.subplots(1, len(qc.columns), figsize=(5 * len(qc.columns), 4), squeeze=False)
for ax, column in zip(axes[0], qc.columns):
    for name, color in [('Mean', '#0072B2'), ('BABEL', '#E69F00')]:
        ax.scatter(qc[column], r[name], s=7, alpha=.2, label=name, color=color)
    ax.set(xlabel=column, ylabel='Officiële Pearson per cel')
    ax.legend()
show('03_rna_detection_depth')
r.join(qc).to_csv(OUT / 'per_cell_diagnostics.csv', index_label='cell_id')


For both models, a higher Pearson correlates with more detected RNA-genes and a higher RNA-count total.

In [ ]:
fig, axes = plt.subplots(
    2, len(qc.columns),
    figsize=(6 * len(qc.columns), 8),
    sharex='col', sharey=True, squeeze=False
)

for row, (name, color) in enumerate([
    ('Mean', '#0072B2'),
    ('BABEL', '#E69F00')
]):
    for col, column in enumerate(qc.columns):
        ax = axes[row, col]
        ax.scatter(qc[column], r[name], s=7, alpha=.3, color=color)
        ax.set(
            title='Mean per gene' if name == 'Mean' else 'BABEL',
            xlabel=column,
            ylabel='Officiële Pearson per cel'
        )

show('04_rna_detection_depth_separate')

## Removing more top ranking genes

In [ ]:
ranking = np.argsort(contribution)[::-1]
removal = []

for k in [0, 10, 100, 500, 1000]:
    keep = np.sort(ranking[k:])
    values = []

    for start in range(0, Y.shape[0], 128):
        sl = slice(start, start + 128)
        y = unit_rows(dense(Y[sl][:, keep]))
        p = unit_rows(dense(P['Mean'][sl][:, keep]))
        values.extend((y * p).sum(axis=1))

    removal.append({
        'Weggelaten genen': k,
        'Gemiddelde Pearson': np.mean(values)
    })

removal = pd.DataFrame(removal)
assert np.isclose(removal.iloc[0, 1], r['Mean'].mean(), atol=1e-5)
display(removal)

plt.figure(figsize=(6, 4))
plt.plot(removal.iloc[:, 0], removal.iloc[:, 1], 'o-')
plt.xlabel('Aantal weggelaten topgenen')
plt.ylabel('Gemiddelde baseline-Pearson')
show('05_remove_top_genes')

## Contribution of zero counts to the Pearson score

In [ ]:
parts = []

for start in range(0, Y.shape[0], 128):
    sl = slice(start, start + 128)
    y = dense(Y[sl])
    p = dense(P['Mean'][sl])
    terms = unit_rows(y) * unit_rows(p)

    parts.append(np.column_stack([
        (terms * (y == 0)).sum(axis=1),
        (terms * (y > 0)).sum(axis=1),
        (y == 0).mean(axis=1)
    ]))

parts = pd.DataFrame(
    np.vstack(parts), index=r.index,
    columns=['Bijdrage nulmetingen', 'Bijdrage positieve metingen',
             'Fractie nulmetingen']
)
assert np.allclose(
    parts.iloc[:, :2].sum(axis=1), r['Mean'], atol=1e-5
)

display(parts.mean().to_frame('Gemiddelde over cellen'))

parts.iloc[:, :2].mean().plot.bar(
    figsize=(6, 4), color=['#0072B2', '#E69F00'], rot=0
)
plt.axhline(0, color='black', lw=1)
plt.ylabel('Bijdrage aan gemiddelde baseline-Pearson')
show('06_zero_positive_contributions')